In [2]:
from pathlib import Path
import sys
import pandas as pd
from dotenv import load_dotenv
import plotly.io as io
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.paths import DATA_RAW_PATH, DATA_OUTPUTS_PATH

In [3]:
df_x = pd.read_excel(DATA_RAW_PATH / 'get_around_delay_analysis.xlsx')
df_x.to_csv(DATA_RAW_PATH / 'get_around_delay_analysis.csv', index=None, header=True)

df_analysis = pd.read_csv(DATA_RAW_PATH / 'get_around_delay_analysis.csv')


print(f"The DF for analysis : ")
display(df_analysis)


The DF for analysis : 


,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN
...,...,...,...,...,...,...,...
21305,573446,380069,mobile,ended,NaN,573429.0,300.0
21306,573790,341965,mobile,ended,-337.0,NaN,NaN
21307,573791,364890,mobile,ended,144.0,NaN,NaN
21308,574852,362531,connect,ended,-76.0,NaN,NaN


In [4]:
df_analysis['time_delta'] = df_analysis['time_delta_with_previous_rental_in_minutes']
df_analysis['prev_id'] = df_analysis['previous_ended_rental_id']
df_analysis['delay_checkout'] = df_analysis['delay_at_checkout_in_minutes']

df_analysis = df_analysis.drop(columns=['time_delta_with_previous_rental_in_minutes','previous_ended_rental_id','delay_at_checkout_in_minutes'])

df_analysis.head()

,rental_id,car_id,checkin_type,state,time_delta,prev_id,delay_checkout
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,NaN,NaN,-81.0
2,508131,359049,connect,ended,NaN,NaN,70.0
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


# Schema validation

In [5]:
print('DF Analysis:')
display(df_analysis.info())


DF Analysis:
<class 'pandas.DataFrame'>
RangeIndex: 21310 entries, 0 to 21309
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   rental_id       21310 non-null  int64  
 1   car_id          21310 non-null  int64  
 2   checkin_type    21310 non-null  str    
 3   state           21310 non-null  str    
 4   time_delta      1841 non-null   float64
 5   prev_id         1841 non-null   float64
 6   delay_checkout  16346 non-null  float64
dtypes: float64(3), int64(2), str(2)
memory usage: 1.1 MB


None

In [6]:
df_analysis['prev_id'] = df_analysis['prev_id'].astype('Int64')

display(df_analysis.info())

<class 'pandas.DataFrame'>
RangeIndex: 21310 entries, 0 to 21309
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   rental_id       21310 non-null  int64  
 1   car_id          21310 non-null  int64  
 2   checkin_type    21310 non-null  str    
 3   state           21310 non-null  str    
 4   time_delta      1841 non-null   float64
 5   prev_id         1841 non-null   Int64  
 6   delay_checkout  16346 non-null  float64
dtypes: Int64(1), float64(2), int64(2), str(2)
memory usage: 1.2 MB


None

# Duplicates

In [7]:
df_analysis.duplicated().sum()

np.int64(0)

# Missing values

In [8]:
count = df_analysis.isna().sum().sort_values(ascending=False)
perc = df_analysis.isna().mean().mul(100).round(2).sort_values(ascending=False)

nan_df = pd.concat([count, perc], axis=1)
nan_df.columns = ['NaN count', 'NaN %']

display(nan_df)

,NaN count,NaN %
time_delta,19469,91.36
prev_id,19469,91.36
delay_checkout,4964,23.29
rental_id,0,0.00
car_id,0,0.00
checkin_type,0,0.00
state,0,0.00


In [9]:
canceled = df_analysis[df_analysis['state'] == 'canceled']
cancel_check = canceled[["prev_id", "time_delta", "delay_checkout"]].isna().mean().mul(100).round(2)

cancel_check.name = 'NaN percentage if state is canceled'

display(cancel_check)


prev_id           92.99
time_delta        92.99
delay_checkout    99.97
Name: NaN percentage if state is canceled, dtype: float64

In [10]:
canceled = df_analysis[df_analysis['state'] == 'ended']
cancel_check = canceled[["prev_id", "time_delta", "delay_checkout"]].isna().mean().mul(100).round(2)

cancel_check.name = 'NaN percentage if state is ended'

display(cancel_check)


prev_id           91.07
time_delta        91.07
delay_checkout     9.42
Name: NaN percentage if state is ended, dtype: float64

In [11]:
previous_rental_check = (
    df_analysis["prev_id"].isna()
    ==
    df_analysis["time_delta"].isna()
).mean() * 100

print(f"Percentage of rows where both features are in the same state (both missing or both filled): {previous_rental_check}%")

Percentage of rows where both features are in the same state (both missing or both filled): 100.0%


* The variables prev_id and time_delta exhibit identical missing-value patterns (100% overlap). This indicates that both variables capture the same business process: the existence of a previous completed rental linked to the current rental.

* When these variables are missing, it generally means that no previous rental could be associated with the current rental. Therefore, these missing values should be interpreted as structural missingness rather than data quality issues.

* Similarly, missing values in delay_checkout are strongly associated with canceled rentals. Since canceled rentals do not reach the checkout stage, no checkout delay can be observed. These missing values are therefore expected and should also be treated as structural missingness.

* Consequently, missing values in these variables will be preserved in the raw dataset. Analyses related to rental succession, inter-rental delays, and checkout delays will be conducted on the relevant subset of observations where the required information is available.

# Data validity

In [12]:
col_positive = ['time_delta','prev_id', 'rental_id', 'car_id']

df_notna = df_analysis.dropna(subset=col_positive)

df_notna.shape

(1841, 7)

In [13]:
df_col_positive = (df_notna[col_positive] < 0).mean().mul(100).round(2)
df_col_positive.name = 'Percentage of Negative values'

df_col_positive

time_delta    0.0
prev_id       0.0
rental_id     0.0
car_id        0.0
Name: Percentage of Negative values, dtype: Float64

No data validity issues were detected in the variables describing rental succession. All previous rental identifiers are strictly positive and all recorded inter-rental time gaps are non-negative.

# Outliers

In [14]:
display(df_analysis['time_delta'].describe())
display(df_analysis['delay_checkout'].describe())

count    1841.000000
mean      279.288430
std       254.594486
min         0.000000
25%        60.000000
50%       180.000000
75%       540.000000
max       720.000000
Name: time_delta, dtype: float64

count    16346.000000
mean        59.701517
std       1002.561635
min     -22433.000000
25%        -36.000000
50%          9.000000
75%         67.000000
max      71084.000000
Name: delay_checkout, dtype: float64

In [15]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Time Delta", "Checkout Delay"]
)

fig.add_trace(
    go.Box(y=df_analysis["time_delta"], name="time_delta"),
    row=1,
    col=1
)

fig.add_trace(
    go.Box(y=df_analysis["delay_checkout"], name="delay_checkout"),
    row=1,
    col=2
)

fig.update_layout(height=500, width=900)

fig.show()

In [16]:
print(f"Number of checkout delay > 1 day : {(df_analysis['delay_checkout'] > 1440).sum()} rentals on {df_analysis.shape[0]}")
print()
print(f"Number of checkout delay < 1 day : {(df_analysis['delay_checkout'] < -1440).sum()} rentals on {df_analysis.shape[0]}")



Number of checkout delay > 1 day : 188 rentals on 21310

Number of checkout delay < 1 day : 41 rentals on 21310


In [17]:
df_analysis_clean = df_analysis[
    df_analysis["delay_checkout"].between(-1440, 1440) 
    | 
    df_analysis['delay_checkout'].isna()
    ]

print(f"Number of checkout delay > 1 day : {(df_analysis_clean['delay_checkout'] > 1440).sum()} rentals on {df_analysis_clean.shape[0]}")
print()
print(f"Number of checkout delay < 1 day : {(df_analysis_clean['delay_checkout'] < -1440).sum()} rentals on {df_analysis_clean.shape[0]}")

Number of checkout delay > 1 day : 0 rentals on 21081

Number of checkout delay < 1 day : 0 rentals on 21081


# Analysis

**DF init**

In [37]:
# Main dataset with structural NaNs
df_analysis

# Successive rentals analysis (supresses also nans for prev_id)
df_chain = df_analysis.loc[df_analysis['time_delta'].notna()].copy()

# Check-in delay analysis
df_delay = df_analysis.loc[df_analysis['delay_checkout'].notna()].copy()



### 1. Which share of owner's revenue would be affected ?

In [52]:
threshold = 120

affected = df_chain.loc[df_chain["time_delta"] < threshold].copy()

perc_affected = (affected.shape[0] / df_chain.shape[0]) * 100
print(f"Percentage of affected owner's revenue for a threshold of {threshold} : {perc_affected:.2f} %")

Percentage of affected owner's revenue for a threshold of 120 : 36.18 %


### 2. How many rentals are affected depending on threshold ?

In [42]:
threshold = range(0, 721, 30)

result = []

for t in threshold:
    
    n_affected = (df_chain['time_delta'] < t).sum()
    
    result.append(
        {'threshold':t,
         'affected_rentals':n_affected
         }
    )

affected_df = pd.DataFrame(result)
affected_df.head(10)


,threshold,affected_rentals
0,0,0
1,30,279
2,60,401
3,90,584
4,120,666
5,150,803
6,180,870
7,210,953
8,240,1001
9,270,1068


In [53]:
fig = px.line(
    affected_df,
    x="threshold",
    y="affected_rentals",
    markers=True
)

fig.show()

The number of affected rentals increases continuously as the threshold grows.

This is expected because:

* a larger buffer protects more future rentals,
* but also removes more booking opportunities.

Business implication:

        Increasing the threshold improves operational safety but reduces inventory availability and potentially owner revenue.

This curve represents the cost of the feature.

### 3. Scope : all cars or only Connect cars ?

In [54]:
df_analysis["checkin_type"].value_counts()

checkin_type
mobile     17003
connect     4307
Name: count, dtype: int64

In [57]:
df_connect = df_chain.loc[
    df_chain["checkin_type"] == "connect"
]

affected_connect = (df_connect["time_delta"] < 120).sum()

df_mobile = df_chain.loc[
    df_chain["checkin_type"] == "mobile"
]

affected_mobile = (df_mobile["time_delta"] < 120).sum()

print(f"Total affected rentals by Connect cars : {affected_connect}")
print()
print(f"Total affected rentals by Mobile cars : {affected_mobile}")

Total affected rentals by Connect cars : 295

Total affected rentals by Mobile cars : 371


### 4. How often are drivers late?

In [61]:
late_rate = (
    (df_delay["delay_checkout"] > 0)
    .mean()
    * 100
)

print(f"Drivers are late : {late_rate:.2f} % of a time")

Drivers are late : 57.53 % of a time


In [64]:
display(df_delay["delay_checkout"].describe())

display(df_delay["delay_checkout"].quantile(
    [0.5, 0.75, 0.9, 0.95]
))

count    16346.000000
mean        59.701517
std       1002.561635
min     -22433.000000
25%        -36.000000
50%          9.000000
75%         67.000000
max      71084.000000
Name: delay_checkout, dtype: float64

0.50      9.00
0.75     67.00
0.90    193.00
0.95    397.75
Name: delay_checkout, dtype: float64

### 5. How often does it impact the next driver?

In [70]:
df_chain["impacts_next_driver"] = (
    df_chain["delay_checkout"]
    >
    df_chain["time_delta"]
)

print(f"Total number of impacted next driver : {df_chain['impacts_next_driver'].sum()}")

perc_impac = (df_chain["impacts_next_driver"].mean()*100)

print(f"Percentage of impacted next driver : {perc_impac:.2f} %")



Total number of impacted next driver : 270
Percentage of impacted next driver : 14.67 %


### 6. How many problematic cases would a threshold solve?

In [73]:
threshold = 120

solved = (
    (df_chain["time_delta"] + threshold) > df_chain["delay_checkout"]
)

print(f"Total number of problematic cases solved with a threshold of {threshold} : {solved.sum()}")

Total number of problematic cases solved with a threshold of 120 : 1418


In [79]:
thresholds = range(0, 721, 30)

results = []

for t in thresholds:

    solved = (
        (df_chain['time_delta'] + t) > df_chain['delay_checkout']
    )

    perc_solved = solved.sum() / df_chain.shape[0] * 100

    results.append(
        {'Threshold': t,
         'Total solved': solved.sum(),
         'Percentage solved %': perc_solved.round(2)}
    )

solved_df = pd.DataFrame(results)

solved_df.head(10)


,Threshold,Total solved,Percentage solved %
0,0,1242,67.46
1,30,1324,71.92
2,60,1373,74.58
3,90,1405,76.32
4,120,1418,77.02
5,150,1439,78.16
6,180,1446,78.54
7,210,1453,78.92
8,240,1457,79.14
9,270,1463,79.47


In [83]:
import plotly.graph_objects as go

fig = go.Figure()

# Bar plot
fig.add_trace(
    go.Bar(
        x=solved_df["Threshold"],
        y=solved_df["Total solved"],
        name="Total solved"
    )
)

# Line plot on secondary axis
fig.add_trace(
    go.Scatter(
        x=solved_df["Threshold"],
        y=solved_df["Percentage solved %"],
        mode="lines+markers",
        name="% solved",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Problematic cases solved by threshold",
    xaxis_title="Threshold (minutes)",

    yaxis=dict(
        title="Total solved rentals"
    ),

    yaxis2=dict(
        title="% solved",
        overlaying="y",
        side="right"
    ),

    hovermode="x unified"
)

fig.show()

The graph shows a rapid increase in solved cases for small thresholds.

Observed values:

| Threshold | % solved |
| --------- | -------- |
| 0 min     | 67.5%    |
| 120 min   | 77.0%    |
| 240 min   | 79.1%    |
| 720 min   | 79.8%    |


Key insight:

* Even without any threshold, about two-thirds of problematic situations are naturally avoided because most delays are smaller than the existing gaps between rentals.

* Adding a 120-minute threshold increases the protection rate from approximately 67% to 77%.

* After that point, improvements become marginal.


In [85]:
solved_df["pct_slope"] = (
    (solved_df["Percentage solved %"].diff() / solved_df["Threshold"].diff())
    .round(2)
)

solved_df[["Threshold", "Percentage solved %", "pct_slope"]]



,Threshold,Percentage solved %,pct_slope
0,0,67.46,NaN
1,30,71.92,0.15
2,60,74.58,0.09
3,90,76.32,0.06
4,120,77.02,0.02
5,150,78.16,0.04
6,180,78.54,0.01
7,210,78.92,0.01
8,240,79.14,0.01
9,270,79.47,0.01


In [99]:
fig = px.line(
    solved_df,
    x="Threshold",
    y="pct_slope",
    markers=True,
    title="Slope of threshold's benefit"
)

fig.show()

### 7. Business cost

In [91]:
comparison_df = (
    solved_df.merge(
        affected_df,
        left_on="Threshold",
        right_on="threshold"
    )
)

display(comparison_df.head())


,Threshold,Total solved,Percentage solved %,pct_slope,threshold,affected_rentals
0,0,1242,67.46,NaN,0,0
1,30,1324,71.92,0.15,30,279
2,60,1373,74.58,0.09,60,401
3,90,1405,76.32,0.06,90,584
4,120,1418,77.02,0.02,120,666


In [95]:
comparison_df["Affected rentals %"] = (comparison_df["affected_rentals"] / df_chain.shape[0] * 100).round(2)

comparison_df["Solved per affected rental"] = (comparison_df["Total solved"] / comparison_df["affected_rentals"]).round(2)

display(comparison_df.sort_values("Solved per affected rental",ascending=False).head())

fig = px.line(
    comparison_df,
    x="Threshold",
    y=[
        "Percentage solved %",
        "Affected rentals %"
    ],
    markers=True
)

fig.show()



,Threshold,Total solved,Percentage solved %,pct_slope,threshold,affected_rentals,Affected rentals %,Solved per affected rental
0,0,1242,67.46,NaN,0,0,0.00,inf
1,30,1324,71.92,0.15,30,279,15.15,4.75
2,60,1373,74.58,0.09,60,401,21.78,3.42
3,90,1405,76.32,0.06,90,584,31.72,2.41
4,120,1418,77.02,0.02,120,666,36.18,2.13
